In [3]:
import os
import numpy as np
import scipy.stats as stats
import torch
import copy
from torch import nn
from torch import optim
import torch.nn.functional as F
import torchvision
from torchvision import datasets
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

In [4]:
with_cuda = torch.cuda.is_available()
if with_cuda:
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

# Text prediction

## The Penn-TreeBank (PTB) dataset

**Question 1**

Take a look at the files `ptb_test.txt`, `ptb_valid.txt`, `ptb_train.txt`. What are the differences with a real text? Why?

**Question 2**

We want to perform text prediction. For that, we have two choices: 
1. character prediction;
2. word prediction.

Specify the format of the input and the output of our model: how should we encode them?

On lit le texte séquentiellement et on attribut des chiffres aux mots que l'on rencontre. Pour les transformers, ce qui est utilisé est de contruire de manière automatique des tokens ou les choses sont découpées en fonction des groupes de lettre les p^lus fréquents. On spécifie un nombre de tokens. 

**Question 3**

We want to replace the characters/words by tokens. For that, we build the classes `Dictionary`, `CorpusWords` and `CorpusChars`.

The class `Dictionary` has attributes:
* a dictionary `word2idx`, which maps words (or characters) to tokens;
* a list `idx2word`, which maps tokens to words (or characters);
* a dictionary `nb_occ`, which contains the number of occurrences of each word (or character).

Write the method `add_word`, which takes a word (or character), and updates the attributes accordingly.

The class `CorpusWords` has attributes:
* the dictionary of words `dictionary`:
* the tokenized train, valid and test texts.

Write the method `tokenize`, which takes a path to a text file and:
1. reads it sequentially word by word and add the words to the dictionary;
2. reads it sequentially word by word, transforms the words into token (by using the dictionary), and store the tokens into a LongTensor; each "new line" character should also be stored;
3. returns the LongTensor in which the token have been stored.

Build the class `CorpusChars` in the same way (by replacing words by characters).

In [8]:
path = "Users\camdus\Desktop\L3\Deep_learning_3\TP_Deep_Learning\ptb"
class Dictionary:
    def __init__(self):
        self.word2idx = {}
        self.idx2word = []
        self.nb_occ = {}

    def add_word(self, word):
        if self.word2idx.has_key(word):
            self.nb_occ[word]+=1
        else : 
            self.nb_occ[word]=1
            self.word2idx[word]= len(self.word2idx)
            self.idx2word.append(word)
        
    def __len__(self):
        return len(self.idx2word)


class CorpusWords:
    def __init__(self, path):
        self.dictionary = Dictionary() #ce qui nous permet de passer des tokens aux mots et inversement
        self.train = self.tokenize(os.path.join(path, 'ptb_train.txt'))
        self.valid = self.tokenize(os.path.join(path, 'ptb_valid.txt'))
        self.test = self.tokenize(os.path.join(path, 'ptb_test.txt'))

    def tokenize(self, path):
        assert os.path.exists(path)
        with open(path, "r", encoding = "utf8") as f:
            tokens = 0
            for line in f: 
                words = line.split() + ['<eos>']
                tokens += len(words)
                for word in words :   
                        self.dictionary.add_word(word)
        
        with open(path, "r", encoding = "utf8") as f:
            token = 0
            ids = torch.LongTensor(tokens)
            for line in f: 
                words = line.split() + ['<eos>']

                for word in words :   
                        ids[token] = self.dictionary.word2idx[word]
                        token += 1
        return ids
#attention : pour utiliser la méthode add_word avec une variable de type dictionnaire, il faut procéder comme suit : dct.add_word(word), 
#puisque le self est implicit (il y a déjà deux arguments)        
    
class CorpusChars:
    pass

On met $<eos>$ pour signifier la fin d'une phrase. Cela permet d'indiquer au RNN la fin de la phrase et de faire en sorte qu'il ne produise pas de sorties abérantes. 

**Question 4** 

What does the following functions do?

In [5]:
def batchify(data, batch_size):
    nbatches = data.size(0) // batch_size
    data = data.narrow(0, 0, nbatches * batch_size)
    data = data.view(batch_size, -1).t().contiguous()
    
    return data

In [9]:
def get_batch(data_src, bptt, i):
    seq_len = min(bptt, len(data_src) - 1 - i)
    data = data_src[i:i + seq_len]
    target = data_src[i + 1:i + 1 + seq_len].view(-1)
    
    return data, target

On remplace un tenseur d'ordre 1 : $[1, 2, ...,  101,10,....]$ \
en un tableau (tenseur d'ordre 2): $[[1, 2, ... ], ...[101, 10,...], ...[...]]$

## Building a LSTM

**Question 5**

Build a class `ModelRNN`:
* which takes arguments:
  * `ntokens`: the number of different tokens (= the size of the dictionary);
  * `nhid`: the number of neurons in the hidden layer of the LSTM;
  * `nlayers`: the number of stacked LSTMs;
* which has attributes:
  * `rnn`: a `nn.LSTM` initialized with the relevant attributes;
  * exercise: other attributes the are necessary to make it work;
* which implements the method `forward`:
  * exercise: implement it; what should be the arguments and the return values of this function?

In [ ]:
class ModelRNN(nn.Module):
    def __init__(self, ntokens, ninp, nhid, nlayers):
        super(ModelRNN, self).__init__() 

        self.rnn = nn.LSTM(ninp, nhid, nlayers)
        
    def forward(self, input, hidden, current):
        #embedding en entrée et en sortie, qui s'entrainent
        x = self.rnn(input,  len(hidden),  len(current))
        hidden = x[1][0]
        current = x[1][1]

        
    
    def init_hidden(self, batch_size):
        device = next(self.parameters()).device
        return (torch.zeros((self.nlayers, batch_size, self.nhid), device = device),
                torch.zeros((self.nlayers, batch_size, self.nhid), device = device))

class torch.nn.Embedding(num_embeddings, embedding_dim, padding_idx=None, max_norm=None, norm_type=2.0, scale_grad_by_freq=False, sparse=False, _weight=None, _freeze=False, device=None, dtype=None)

**Question 6**

Complete the code in `train_model`

In [7]:
def train_model(train_data, bptt, model, criterion, optimizer, nepochs):
    #List to store loss to visualize
    train_losses = []
    train_acc = []
    start_epoch = 0

    for epoch in range(nepochs):
        train_loss = 0.
        valid_loss = 0.
        correct = 0
        total_pred = 0

        model.train()
        
        current, hidden = model.init_hidden(batch_size)
        
        for batch_idx, i in enumerate(range(0, train_data.size(0) - 1, bptt)):
            data, targets = get_batch(train_data, bptt, i)
            data, targets = data.to(device), targets.to(device)
            
            # Complete

        # calculate average losses
        #train_loss = train_loss/len(data_loader.dataset)
        train_losses.append(train_loss)

        # print losses statistics 
        print('Epoch: {} \tTraining Loss: {:.6f}'.format(
            epoch, train_loss / (batch_idx + 1)))

**Question 7**

Construct the batches for character prediction and train the model.

In [8]:
batch_size = 20
bptt = 35

ninp = 100
nhid = 100
nlayers = 2



**Question 8**

Do the same for word prediction. What is the purpose of the `encoder` and the `decoder` modules?